# Task 8 — Mask-based anomaly segmentation baselines with EoMT

This notebook evaluates EoMT on the anomaly segmentation validation datasets used in Task 7.

The pipeline:
1. mounts Google Drive and prepares the repository paths;
2. extracts the anomaly datasets locally in the Colab runtime;
3. evaluates the selected EoMT checkpoints with MSP, MaxLogit-like, Max Entropy and RbA at `T=1`;
4. saves the raw EoMT outputs locally and reuses them for temperature scaling;
5. exports report-ready CSV and Excel tables to Google Drive.

The intermediate mask/class logits are saved only in the local Colab runtime (`/content/saved_logits`) and are not persisted on Drive.


## 1. Environment setup, Drive and repository paths

In [30]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "MaskArchitectureAnomaly_CourseProject"
EOMT_ROOT = PROJECT_ROOT / "eomt"
EVAL_DIR = PROJECT_ROOT / "eval"
LARGE_FILES = DRIVE_ROOT / "FAIML_project_and_presentation" / "01_Project" / "large_files"
RESULTS_ROOT = LARGE_FILES.parent / "results" / "task8"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

for p in (PROJECT_ROOT, EOMT_ROOT, EVAL_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("PROJECT_ROOT:", PROJECT_ROOT, "| exists:", PROJECT_ROOT.exists())
print("EOMT_ROOT:", EOMT_ROOT, "| exists:", EOMT_ROOT.exists())
print("EVAL_DIR:", EVAL_DIR, "| exists:", EVAL_DIR.exists())
print("RESULTS_ROOT:", RESULTS_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject | exists: True
EOMT_ROOT: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eomt | exists: True
EVAL_DIR: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eval | exists: True
RESULTS_ROOT: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8


## 2. Data, weights and output paths

In [31]:
import zipfile
import glob

WEIGHTS_ROOT = LARGE_FILES / "weights"
ANOMALY_ZIP = LARGE_FILES / "datasets" / "anomaly" / "Anomaly_Validation_Datasets.zip"

# Datasets are extracted to the local Colab disk for faster image loading.
LOCAL_DATA_DIR = Path("/content/anomaly_data")
DATA_ROOT = LOCAL_DATA_DIR / "Validation_Dataset"

# Persistent result files on Drive.
from datetime import datetime
RESULTS_CSV = RESULTS_ROOT / f"eomt_last_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.csv"
EXCEL_OUTPUT_PATH = RESULTS_ROOT / f"task8_results_last_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.xlsx"
TEMP_EXCEL_PATH = RESULTS_ROOT / f"task8_temperature_results_last_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.xlsx"

# Temporary local folder for saved EoMT outputs. This folder is lost when the runtime is reset.
LOCAL_LOGITS_DIR = Path("/content/saved_logits")

print("ANOMALY_ZIP:", ANOMALY_ZIP, "| exists:", ANOMALY_ZIP.exists())
print("WEIGHTS_ROOT:", WEIGHTS_ROOT, "| exists:", WEIGHTS_ROOT.exists())
print("RESULTS_CSV:", RESULTS_CSV)
print("EXCEL_OUTPUT_PATH:", EXCEL_OUTPUT_PATH)
print("TEMP_EXCEL_PATH:", TEMP_EXCEL_PATH)
print("LOCAL_LOGITS_DIR:", LOCAL_LOGITS_DIR)


ANOMALY_ZIP: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/datasets/anomaly/Anomaly_Validation_Datasets.zip | exists: True
WEIGHTS_ROOT: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights | exists: True
RESULTS_CSV: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_last_2026-06-05_08-59.csv
EXCEL_OUTPUT_PATH: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/task8_results_last_2026-06-05_08-59.xlsx
TEMP_EXCEL_PATH: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/task8_temperature_results_last_2026-06-05_08-59.xlsx
LOCAL_LOGITS_DIR: /content/saved_logits


## 3. Extract the anomaly validation datasets

The anomaly zip is extracted to `/content` to avoid repeatedly reading images from Drive during inference.


In [32]:
print("Zip exists:", ANOMALY_ZIP.exists())

if not LOCAL_DATA_DIR.exists():
    print("Extracting zip temporarily to /content...")
    with zipfile.ZipFile(ANOMALY_ZIP, "r") as z:
        z.extractall(LOCAL_DATA_DIR)
    print("Done.")
else:
    print("Already extracted in this runtime.")

print("DATA_ROOT exists:", DATA_ROOT.exists())
if DATA_ROOT.exists():
    for p in sorted(DATA_ROOT.iterdir()):
        print("-", p.name)


Zip exists: True
Extracting zip temporarily to /content...
Done.
DATA_ROOT exists: True
- .DS_Store
- FS_LostFound_full
- RoadAnomaly
- RoadAnomaly21
- RoadObsticle21
- fs_static


## 4. Define anomaly datasets

In [33]:
datasets = {
    "FS_LostFound_full": DATA_ROOT / "FS_LostFound_full" / "images" / "*.png",
    "fs_static": DATA_ROOT / "fs_static" / "images" / "*.jpg",
    "RoadAnomaly": DATA_ROOT / "RoadAnomaly" / "images" / "*.jpg",
    "RoadAnomaly21": DATA_ROOT / "RoadAnomaly21" / "images" / "*.png",
    "RoadObsticle21": DATA_ROOT / "RoadObsticle21" / "images" / "*.webp",
}

for name, pattern in datasets.items():
    files = glob.glob(str(pattern))
    print(name, len(files), "images")


FS_LostFound_full 100 images
fs_static 30 images
RoadAnomaly 60 images
RoadAnomaly21 10 images
RoadObsticle21 30 images


## 5. Select EoMT checkpoints

The project requires the evaluation of three EoMT checkpoints:
- the COCO-trained panoptic checkpoint;
- the Cityscapes-trained semantic checkpoint;
- the head-only baseline COCO-to-Cityscapes checkpoint;
- the fine-tuned COCO-to-Cityscapes checkpoint.


In [43]:
#EOMT_COCO_WEIGHTS = WEIGHTS_ROOT / "eomt_coco.bin"
EOMT_CITYSCAPES_WEIGHTS = WEIGHTS_ROOT / "eomt_cityscapes.bin"
#EOMT_FINETUNED_WEIGHTS = WEIGHTS_ROOT / "finetuned" / "coco_to_cityscapes_stage1_head" / "stage1_weights.bin" # Stage 1 weights, trained for 20 epochs with early stopping
EOMT_FINETUNED_WEIGHTS = WEIGHTS_ROOT / "finetuned" / "coco_to_cityscapes_stage2_unfreeze_last" / "stage2_weights_rerun.bin" # Stage 2 weights, trained for 40 epochs with no early
#EOMT_BASELINE_WEIGHTS = WEIGHTS_ROOT / "finetuned" / "coco_to_cityscapes_head_cache" / "head_only_weights_512.bin"


checkpoints = {
    # "eomt_coco": {
    #     "weights": EOMT_COCO_WEIGHTS,
    #     "preset": "coco",
    # },
    "eomt_cityscapes": {
         "weights": EOMT_CITYSCAPES_WEIGHTS,
         "preset": "cityscapes",
    },
    #  "eomt_baseline": {
    #      "weights": EOMT_BASELINE_WEIGHTS,
    #      "preset": "finetuned",
    #  },
    "eomt_finetuned": {
         "weights": EOMT_FINETUNED_WEIGHTS,
         "preset": "finetuned",
     },
}

for name, cfg in checkpoints.items():
    print(name, "->", cfg["weights"], "| exists:", cfg["weights"].exists())

eomt_finetuned -> /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/finetuned/coco_to_cityscapes_stage2_unfreeze_last/stage2_weights_rerun.bin | exists: True


## 6. Check checkpoint compatibility

This cell verifies that each checkpoint has the expected number of queries and classes for its preset.


In [44]:
import torch

def inspect_ckpt(path):
    sd = torch.load(path, map_location="cpu")
    sd = sd["state_dict"] if isinstance(sd, dict) and "state_dict" in sd else sd
    num_q = num_classes = None
    for k, v in sd.items():
        kk = k.replace("._orig_mod", "").replace("network.", "").replace("module.", "")
        if kk.endswith("q.weight"):
            num_q = v.shape[0]
        if kk.endswith("class_head.weight"):
            num_classes = v.shape[0] - 1
    return num_q, num_classes

PRESET_EXPECTED = {
    "coco": (200, 133),
    "cityscapes": (100, 19),
    "finetuned": (200, 19),
}

for name, cfg in checkpoints.items():
    if not cfg["weights"].exists():
        print(f"{name}: weights not found"); continue
    q, c = inspect_ckpt(cfg["weights"])
    exp_q, exp_c = PRESET_EXPECTED[cfg["preset"]]
    ok = (q == exp_q) and (c == exp_c)
    flag = "OK" if ok else "CHECK THIS PRESET"
    print(f"{name:24s} preset={cfg['preset']:10s} num_q={q} num_classes={c} "
          f"(expected {exp_q}/{exp_c}) -> {flag}")


eomt_finetuned           preset=finetuned  num_q=200 num_classes=19 (expected 200/19) -> OK


## 7. Baseline evaluation at temperature `T = 1`

For each checkpoint and dataset, the model forward pass is executed once per image. The raw EoMT outputs are saved locally as `.pt` files:
- `ml`: mask logits;
- `cl`: query class logits;
- `gt_v`: flattened valid anomaly ground truth;
- `valid`: valid-pixel mask;
- `pattern`: original dataset glob pattern.

The four Task 8 post-hoc methods are computed at `T=1`: MSP, MaxLogit-like, Max Entropy and RbA.

In [45]:
import shutil
import time
from PIL import Image
import numpy as np
from types import SimpleNamespace
from ood_metrics import fpr_at_95_tpr
from sklearn.metrics import average_precision_score
import evalAnomaly_eomt as E 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = True
print("Device:", DEVICE)

# Start from a clean local logits directory and a clean CSV.
if LOCAL_LOGITS_DIR.exists():
    shutil.rmtree(LOCAL_LOGITS_DIR)
LOCAL_LOGITS_DIR.mkdir(parents=True, exist_ok=True)

if RESULTS_CSV.exists():
    RESULTS_CSV.unlink()

methods_baseline = ["msp", "maxlogit", "entropy", "rba"]
target_size = (E.IMG_HEIGHT, E.IMG_WIDTH)


def build_model(weights_path, preset):
    args = SimpleNamespace(
        preset=preset, 
        num_blocks=3, 
        patch_size=16,
        backbone_name="vit_base_patch14_reg4_dinov2",
    )
    args = E.apply_preset(args)
    model = E.build_eomt(args)
    print(f"Loading weights {os.path.basename(str(weights_path))}")
    model = E.load_eomt_weights(model, str(weights_path))
    return model.to(DEVICE).eval(), args

for checkpoint_name, cfg in checkpoints.items():
    if not cfg["weights"].exists():
        print(f"Skip {checkpoint_name}: weights not found")
        continue
        
    model, args = build_model(cfg["weights"], cfg["preset"])

    for dataset_name, pattern in datasets.items():
        paths = sorted(glob.glob(str(pattern)))
        if not paths: continue

        scores_t1 = {m: [] for m in methods_baseline}
        gts_t1 = {m: [] for m in methods_baseline}
        input_pattern_str = str(pattern)

        print(f"\n[FORWARD] {checkpoint_name} -> {dataset_name} ({len(paths)} images)")
        t_ds = time.time()
        
        for idx, path in enumerate(paths):
            img = E.input_transform(Image.open(path).convert("RGB")).unsqueeze(0).float().to(DEVICE)
            
            with torch.no_grad():
                with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=(USE_AMP and DEVICE.type == "cuda")):
                    ml_layers, cl_layers = model(img)
                
                # Move outputs to CPU immediately to keep GPU memory low.
                ml = ml_layers[-1].float().cpu()
                cl = cl_layers[-1].float().cpu()
                
                # Load and align the anomaly ground truth.
                pathGT = E.get_gt_path(path)
                if not os.path.exists(pathGT): continue

                gt = E.convert_gt(np.array(E.target_transform(Image.open(pathGT))), pathGT)
                if 1 not in np.unique(gt): continue
                
                valid = (gt == 0) | (gt == 1)
                gt_v = gt[valid].astype(np.uint8)

                # Save the raw mask-architecture outputs.
                payload = {
                    "ml": ml, "cl": cl, "gt_v": gt_v, "valid": valid, "pattern": input_pattern_str
                }
                torch.save(payload, f"{LOCAL_LOGITS_DIR}/{checkpoint_name}_{dataset_name}_img_{idx}.pt")

                ml_d = ml.to(DEVICE)
                cl_d = cl.to(DEVICE)

                semantic_scores, semantic_probs = E.eomt_to_pixel_scores(
                    ml_d,
                    cl_d,
                    target_size,
                    temperature=1.0,
                )

                for m in methods_baseline:
                    if m == "rba":
                        s = E.compute_rba_score(
                            ml_d,
                            cl_d,
                            target_size,
                            temperature=1.0,
                        )
                    else:
                        s = E.compute_eomt_anomaly_score(
                            semantic_scores,
                            semantic_probs,
                            method=m,
                        )
                    
                    s_np = s.squeeze(0).float().cpu().numpy()[valid].astype(np.float32)
                    scores_t1[m].append(s_np)
                    gts_t1[m].append(gt_v)

        print(f"Dataset processed in {time.time() - t_ds:.1f}s")

        # Write T=1.0 results to the Drive CSV.
        for m in methods_baseline:
            if not gts_t1[m]: continue

            label = np.concatenate(gts_t1[m])
            out = np.concatenate(scores_t1[m])
            auprc = average_precision_score(label, out)
            fpr95 = fpr_at_95_tpr(out, label)

            args.checkpoint_name = checkpoint_name
            args.weights = str(cfg["weights"])
            args.input = [input_pattern_str]
            args.method = m
            args.temperature = 1.0
            args.results_csv = str(RESULTS_CSV)
            
            E.save_csv(args, auprc, fpr95, num_images=len(gts_t1[m]), num_pixels=len(label), num_anomaly_pixels=int(label.sum()))
            print(f"  ->  T=1.0 | {m:8s} -> AUPRC={auprc*100:.2f}  FPR95={fpr95*100:.2f}")

    del model
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

print("\nBaseline evaluation completed. Local logits saved in:", LOCAL_LOGITS_DIR)

Device: cuda
Loading weights stage2_weights_rerun.bin
Interpolating pos_embed from (1, 1024, 768) to (1, 2048, 768)
Checkpoint caricato: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/finetuned/coco_to_cityscapes_stage2_unfreeze_last/stage2_weights_rerun.bin
Numero chiavi checkpoint: 197

[FORWARD] eomt_finetuned -> FS_LostFound_full (100 images)
Dataset processed in 21.1s
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_last_2026-06-05_08-59.csv
  ->  T=1.0 | msp      -> AUPRC=9.52  FPR95=59.63
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_last_2026-06-05_08-59.csv
  ->  T=1.0 | maxlogit -> AUPRC=37.87  FPR95=20.65
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_last_2026-06-05_08-59.csv
  ->  T=1.0 | entropy  -> AUPRC=16.35  FPR95=59.92
Risultati salvati in: /content/drive/M

## 8. Temperature scaling from saved logits

This cell reuses the locally saved EoMT outputs and tests several temperatures without running the network again.

Temperature is applied to the query class logits before the softmax. For EoMT this can also affect the MaxLogit-like score, because the pixel-level semantic scores are reconstructed from temperature-scaled query probabilities.

In [46]:
calib_temperatures = [0.5, 0.75, 1.1]
calib_methods = ["msp", "maxlogit", "entropy", "rba"]

target_size = (E.IMG_HEIGHT, E.IMG_WIDTH)

print("Starting offline temperature scaling from saved local logits...")

for checkpoint_name, cfg in checkpoints.items():
    if not cfg["weights"].exists(): continue
    
    # Create a dummy args object to reuse E.save_csv.
    from types import SimpleNamespace
    dummy_args = SimpleNamespace(
        preset=cfg["preset"], 
        num_blocks=3, 
        patch_size=16,
        backbone_name="vit_base_patch14_reg4_dinov2",
        checkpoint_name=checkpoint_name, 
        weights=str(cfg["weights"]),
        results_csv=str(RESULTS_CSV)
    )
    dummy_args = E.apply_preset(dummy_args)

    for dataset_name in datasets.keys():
        saved_files = sorted(glob.glob(f"{LOCAL_LOGITS_DIR}/{checkpoint_name}_{dataset_name}_img_*.pt"))
        if not saved_files: continue

        print(f"\n[TEMPERATURE GRID] {checkpoint_name} -> {dataset_name} ({len(saved_files)} images)")
        t_start = time.time()
        
        # Initialize containers for dataset-level aggregation.
        all_gts = []
        # Create a dictionary to collect scores for each method-temperature pair.
        grid_scores = {(m, T): [] for m in calib_methods for T in calib_temperatures}
        input_pattern_str = ""

        # Read each saved file only once.
        for file_path in saved_files:
            # Load tensors and move them to the active device.
            payload = torch.load(file_path, map_location=DEVICE, weights_only=False) 

            # Post-processing can be done on GPU if available.
            ml = payload["ml"].to(DEVICE)
            cl = payload["cl"].to(DEVICE)
            gt_v = payload["gt_v"]
            valid = payload["valid"]
            input_pattern_str = payload["pattern"]
            
            all_gts.append(gt_v)

            with torch.no_grad():
                for T in calib_temperatures:
                    semantic_scores, semantic_probs = E.eomt_to_pixel_scores(
                        ml,
                        cl,
                        target_size,
                        temperature=T,
                    )

                    for m in calib_methods:
                        if m == "rba":
                            s = E.compute_rba_score(
                                ml,
                                cl,
                                target_size,
                                temperature=T,
                            )
                        else:
                            s = E.compute_eomt_anomaly_score(
                                semantic_scores,
                                semantic_probs,
                                method=m,
                            )
                        s_np = s.squeeze(0).float().cpu().numpy()[valid].astype(np.float32)
                        grid_scores[(m, T)].append(s_np)

        # Aggregate metrics over the full dataset.
        if all_gts:
            flat_gts = np.concatenate(all_gts)
            
            for m in calib_methods:
                for T in calib_temperatures:
                    flat_scores = np.concatenate(grid_scores[(m, T)])

                    auprc = average_precision_score(flat_gts, flat_scores)
                    fpr95 = fpr_at_95_tpr(flat_scores, flat_gts)

                    dummy_args.method = m
                    dummy_args.temperature = T
                    dummy_args.input = [input_pattern_str]
                    
                    E.save_csv(dummy_args, auprc, fpr95, num_images=len(all_gts), num_pixels=len(flat_gts), num_anomaly_pixels=int(flat_gts.sum()))
                    print(f"  -> method: {m:8s} | T = {T:<4} -> AUPRC: {auprc*100:.2f}% | FPR95: {fpr95*100:.2f}%")
            
            print(f"  Dataset completed in {time.time() - t_start:.1f}s")

print("\nTemperature scaling completed.")

Starting offline temperature scaling from saved local logits...

[TEMPERATURE GRID] eomt_finetuned -> FS_LostFound_full (99 images)
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_last_2026-06-05_08-59.csv
  -> method: msp      | T = 0.5  -> AUPRC: 1.34% | FPR95: 43.02%
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_last_2026-06-05_08-59.csv
  -> method: msp      | T = 0.75 -> AUPRC: 4.91% | FPR95: 46.23%
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_last_2026-06-05_08-59.csv
  -> method: msp      | T = 1.1  -> AUPRC: 11.09% | FPR95: 63.33%
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_last_2026-06-05_08-59.csv
  -> method: maxlogit | T = 0.5  -> AUPRC: 44.95% | FPR95: 25.52%
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01

## 9. Load and inspect the result CSV

In [47]:
import pandas as pd

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)
    display(df.tail(30))
else:
    print("CSV not found:", RESULTS_CSV)


,checkpoint_name,preset,weights,input,method,temperature,num_classes,num_q,num_blocks,img_height,img_width,AUPRC,FPR95,num_images,num_pixels,num_anomaly_pixels
50,eomt_finetuned,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,entropy,0.50,19,200,3,512,1024,54.308684,47.225411,60,31457280,3098336
51,eomt_finetuned,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,entropy,0.75,19,200,3,512,1024,68.437250,41.278346,60,31457280,3098336
52,eomt_finetuned,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,entropy,1.10,19,200,3,512,1024,77.342645,34.539763,60,31457280,3098336
53,eomt_finetuned,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,rba,0.50,19,200,3,512,1024,62.635642,33.228414,60,31457280,3098336
54,eomt_finetuned,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,rba,0.75,19,200,3,512,1024,60.029406,67.439514,60,31457280,3098336
55,eomt_finetuned,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,rba,1.10,19,200,3,512,1024,52.992179,74.917767,60,31457280,3098336
56,eomt_finetuned,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,msp,0.50,19,200,3,512,1024,44.177953,32.024938,10,5060302,749109
57,eomt_finetuned,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,msp,0.75,19,200,3,512,1024,53.917964,30.045071,10,5060302,749109
58,eomt_finetuned,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,msp,1.10,19,200,3,512,1024,67.477393,29.768257,10,5060302,749109
59,eomt_finetuned,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,maxlogit,0.50,19,200,3,512,1024,58.499816,92.990015,10,5060302,749109


## 10. Pivot tables for quick inspection

In [48]:
if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)

    def infer_dataset_name(input_path):
        p = str(input_path)
        for name in datasets.keys():
            if name in p:
                return name
        return "unknown"

    df["dataset"] = df["input"].apply(infer_dataset_name)

    pivot_auprc = df.pivot_table(
        index=["checkpoint_name", "preset", "method", "temperature"],
        columns="dataset",
        values="AUPRC",
        aggfunc="last",
    )

    pivot_fpr95 = df.pivot_table(
        index=["checkpoint_name", "preset", "method", "temperature"],
        columns="dataset",
        values="FPR95",
        aggfunc="last",
    )

    print("AUPRC")
    display(pivot_auprc)

    print("FPR95")
    display(pivot_fpr95)
else:
    print("CSV not created yet.")

AUPRC


dataset                                         FS_LostFound_full  \
checkpoint_name preset    method   temperature                      
eomt_finetuned  finetuned entropy  0.50                  1.963021   
                                   0.75                  9.085649   
                                   1.00                 16.352367   
                                   1.10                 17.119916   
                          maxlogit 0.50                 44.945205   
                                   0.75                 41.340891   
                                   1.00                 37.873526   
                                   1.10                 36.819943   
                          msp      0.50                  1.341839   
                                   0.75                  4.907287   
                                   1.00                  9.516138   
                                   1.10                 11.085521   
                          rba      0.50                 46.348246   
                                   0.75                 42.159996   
                                   1.00                 38.623897   
                                   1.10                 36.896682   

dataset                                         RoadAnomaly  RoadObsticle21  \
checkpoint_name preset    method   temperature                                
eomt_finetuned  finetuned entropy  0.50           53.031206       67.116460   
                                   0.75           67.300900       80.896468   
                                   1.00           76.284487       85.729519   
                                   1.10           77.363576       86.448505   
                          maxlogit 0.50           58.499816       90.207307   
                                   0.75           57.388083       85.372193   
                                   1.00           57.122110       77.029131   
                                   1.10           57.729738       72.820932   
                          msp      0.50           44.177953       55.598863   
                                   0.75           53.917964       71.623634   
                                   1.00           65.023422       79.912185   
                                   1.10           67.477393       80.992920   
                          rba      0.50           49.276562       89.256465   
                                   0.75           44.390120       81.634581   
                                   1.00           39.662974       66.390970   
                                   1.10           36.784103       59.218626   

dataset                                         fs_static  
checkpoint_name preset    method   temperature             
eomt_finetuned  finetuned entropy  0.50         30.111606  
                                   0.75         56.035967  
                                   1.00         69.876704  
                                   1.10         72.088896  
                          maxlogit 0.50         89.052223  
                                   0.75         88.461295  
                                   1.00         86.524382  
                                   1.10         85.457097  
                          msp      0.50         19.326393  
                                   0.75         38.065141  
                                   1.00         55.206491  
                                   1.10         59.423742  
                          rba      0.50         90.878537  
                                   0.75         89.270208  
                                   1.00         84.439703  
                                   1.10         81.695715

FPR95


dataset                                         FS_LostFound_full  \
checkpoint_name preset    method   temperature                      
eomt_finetuned  finetuned entropy  0.50                 42.911856   
                                   0.75                 46.913717   
                                   1.00                 59.918878   
                                   1.10                 63.099765   
                          maxlogit 0.50                 25.517302   
                                   0.75                 24.379584   
                                   1.00                 20.646067   
                                   1.10                 19.838178   
                          msp      0.50                 43.020899   
                                   0.75                 46.228320   
                                   1.00                 59.627170   
                                   1.10                 63.330007   
                          rba      0.50                 35.544114   
                                   0.75                 54.271701   
                                   1.00                 47.855935   
                                   1.10                 50.741018   

dataset                                         RoadAnomaly  RoadObsticle21  \
checkpoint_name preset    method   temperature                                
eomt_finetuned  finetuned entropy  0.50           33.856197        8.481188   
                                   0.75           30.309499        5.886613   
                                   1.00           29.381519        3.630977   
                                   1.10           29.399426        3.104671   
                          maxlogit 0.50           92.990015        0.660179   
                                   0.75           91.430174        0.735966   
                                   1.00           90.301571        2.367339   
                                   1.10           89.883867        2.426813   
                          msp      0.50           32.024938        8.782528   
                                   0.75           30.045071        6.317796   
                                   1.00           29.234717        4.486844   
                                   1.10           29.768257        3.901271   
                          rba      0.50           81.569440        2.296852   
                                   0.75           80.656491        2.372431   
                                   1.00           85.172271        2.948130   
                                   1.10           86.728662       97.588816   

dataset                                         fs_static  
checkpoint_name preset    method   temperature             
eomt_finetuned  finetuned entropy  0.50         41.855397  
                                   0.75         32.862725  
                                   1.00         26.574709  
                                   1.10         24.487320  
                          maxlogit 0.50          7.514946  
                                   0.75          7.568237  
                                   1.00          7.463075  
                                   1.10          7.422908  
                          msp      0.50         42.397035  
                                   0.75         33.455640  
                                   1.00         27.590176  
                                   1.10         25.670701  
                          rba      0.50          2.923171  
                                   0.75          2.740639  
                                   1.00          3.040154  
                                   1.10          3.919603

## 11. Export the `T=1` baseline table to Excel

The mIoU column is optional. Fill the `MIOU_BY_CHECKPOINT` dictionary with the values obtained in Tasks 4/5 if you want them to appear in the final table.


In [49]:
!pip install xlsxwriter

In [50]:
import os

DATASET_COLS = {
    "RoadAnomaly21": "SMIYC RA-21",
    "RoadObsticle21": "SMIYC RO-21",
    "FS_LostFound_full": "FS L&F",
    "fs_static": "FS Static",
    "RoadAnomaly": "Road Anomaly",
}
MODEL_LABELS = {
    "eomt_coco": "EoMT COCO",
    "eomt_cityscapes": "EoMT Cityscapes",
    "eomt_finetuned": "EoMT finetuned",
    "eomt_baseline": "EoMT baseline"
}
METHOD_LABELS = {"msp": "MSP", "maxlogit": "MaxLogit", "entropy": "Max Entropy", "rba": "RbA"}

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)

    search_keys = sorted(DATASET_COLS.keys(), key=len, reverse=True)

    def infer_dataset_name(input_path):
        p = str(input_path)
        for name in search_keys:
            if name in p:
                return name
        return "unknown"

    df["dataset"] = df["input"].apply(infer_dataset_name)
    df = df[df["temperature"] == 1.0]

    col_tuples = [(lbl, metric) for lbl in DATASET_COLS.values() for metric in ("AuPRC", "FPR95")]
    data = {ct: [] for ct in col_tuples}
    index_tuples = []

    ckpt_list = list(checkpoints.keys()) if ('checkpoints' in locals() or 'checkpoints' in globals()) else list(MODEL_LABELS.keys())

    for ckpt in ckpt_list:            
        for m in METHOD_LABELS:              
            sub = df[(df["checkpoint_name"] == ckpt) & (df["method"] == m)]
            if sub.empty:
                continue
            index_tuples.append((MODEL_LABELS.get(ckpt, ckpt), METHOD_LABELS[m]))
            for ds_raw, ds_label in DATASET_COLS.items():
                r = sub[sub["dataset"] == ds_raw]
                a = round(float(r["AUPRC"].iloc[-1]), 2) if not r.empty else None
                f = round(float(r["FPR95"].iloc[-1]), 2) if not r.empty else None
                data[(ds_label, "AuPRC")].append(a)
                data[(ds_label, "FPR95")].append(f)

    table = pd.DataFrame(
        data,
        index=pd.MultiIndex.from_tuples(index_tuples, names=["Model", "Method"]),
    )
    table.columns = pd.MultiIndex.from_tuples(table.columns)

    with pd.ExcelWriter(EXCEL_OUTPUT_PATH, engine="xlsxwriter") as writer:
        table.to_excel(writer, sheet_name="Task8")

    print("Saved Excel table:", EXCEL_OUTPUT_PATH)
    display(table)
else:
    print("CSV not found. Run the evaluation first.")

Saved Excel table: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/task8_results_last_2026-06-05_08-59.xlsx


SMIYC RA-21        SMIYC RO-21       FS L&F         \
                                 AuPRC  FPR95       AuPRC FPR95  AuPRC  FPR95   
Model          Method                                                           
EoMT finetuned MSP               65.02  29.23       79.91  4.49   9.52  59.63   
               MaxLogit          57.12  90.30       77.03  2.37  37.87  20.65   
               Max Entropy       76.28  29.38       85.73  3.63  16.35  59.92   
               RbA               39.66  85.17       66.39  2.95  38.62  47.86   

                           FS Static        Road Anomaly         
                               AuPRC  FPR95        AuPRC  FPR95  
Model          Method                                            
EoMT finetuned MSP             55.21  27.59        66.55  37.25  
               MaxLogit        86.52   7.46        61.26  35.30  
               Max Entropy     69.88  26.57        75.66  36.57  
               RbA             84.44   3.04        55.57  72.77

## 12. Export temperature-scaling tables to Excel

For each checkpoint, this cell creates a sheet containing the results at each tested temperature plus a summary row. The `best t` row reports the best value independently for each metric, so it should be interpreted as the best achievable value across the tested temperatures, not necessarily as a single shared temperature for AuPRC and FPR95.

In [51]:
# Export temperature-scaling results to Excel.

import pandas as pd
import os

TEMP_METHOD_LABELS = {"msp": "MSP", "entropy": "Max Entropy", "rba": "RbA", "maxlogit" : "Max Logit" }

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)

    search_keys = sorted(DATASET_COLS.keys(), key=len, reverse=True)

    def infer_dataset_name(input_path):
        p = str(input_path)
        for name in search_keys:
            if name in p:
                return name
        return "unknown"

    df["dataset"] = df["input"].apply(infer_dataset_name)

    col_tuples = [("", "mIoU")] + [(lbl, met) for lbl in DATASET_COLS.values()
                                   for met in ("AuPRC", "FPR95")]

    def vals_at_temp(g, T, ds_raw):
        r = g[(g["dataset"] == ds_raw) & (g["temperature"] == T)]
        if r.empty:
            return None, None
        return round(float(r["AUPRC"].iloc[-1]), 2), round(float(r["FPR95"].iloc[-1]), 2)

    def vals_best(g, ds_raw): 
        r = g[g["dataset"] == ds_raw]
        if r.empty:
            return None, None
        return round(float(r["AUPRC"].max()), 2), round(float(r["FPR95"].min()), 2)

    sheets = {}
    
    ckpt_list = list(checkpoints.keys()) if ('checkpoints' in locals() or 'checkpoints' in globals()) else ["eomt_coco", "eomt_cityscapes", "eomt_finetuned"]

    for ckpt in ckpt_list:
        data = {ct: [] for ct in col_tuples}
        index_labels = []
        for m, mlabel in TEMP_METHOD_LABELS.items():
            g = df[(df["checkpoint_name"] == ckpt) & (df["method"] == m)]
            if g.empty or g["temperature"].nunique() <= 1:
                continue
            temps = sorted(g["temperature"].unique())
            rows = []
            if 1.0 in temps:
                rows.append((mlabel, lambda ds, g=g: vals_at_temp(g, 1.0, ds)))
            for T in [t for t in temps if t != 1.0]:
                rows.append((f"{mlabel}(t={T})", lambda ds, g=g, T=T: vals_at_temp(g, T, ds)))
            rows.append((f"{mlabel} (best t)", lambda ds, g=g: vals_best(g, ds)))
            for label, getter in rows:
                index_labels.append(label)
                data[("", "mIoU")].append("----")
                for ds_raw, ds_label in DATASET_COLS.items():
                    a, f = getter(ds_raw)
                    data[(ds_label, "AuPRC")].append(a)
                    data[(ds_label, "FPR95")].append(f)
        if index_labels:
            table = pd.DataFrame(data, index=pd.Index(index_labels, name="Method"))
            table.columns = pd.MultiIndex.from_tuples(table.columns)
            sheets[ckpt] = table

    if not sheets:
        print("No temperature-scaling results found. Run the temperature-scaling cell first.")
    else:
        with pd.ExcelWriter(TEMP_EXCEL_PATH, engine="xlsxwriter") as writer:
            for ckpt, table in sheets.items():
                table.to_excel(writer, sheet_name=ckpt[:31])
                print("Saved temperature-scaling Excel table:", TEMP_EXCEL_PATH)
        print("Sheets:", list(sheets.keys()))
        display(list(sheets.values())[-1])
else:
    print("CSV not found. Run the evaluation first.")

Saved temperature-scaling Excel table: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/task8_temperature_results_last_2026-06-05_08-59.xlsx
Sheets: ['eomt_finetuned']


SMIYC RA-21        SMIYC RO-21        FS L&F  \
                      mIoU       AuPRC  FPR95       AuPRC  FPR95  AuPRC   
Method                                                                    
MSP                   ----       65.02  29.23       79.91   4.49   9.52   
MSP(t=0.5)            ----       44.18  32.02       55.60   8.78   1.34   
MSP(t=0.75)           ----       53.92  30.05       71.62   6.32   4.91   
MSP(t=1.1)            ----       67.48  29.77       80.99   3.90  11.09   
MSP (best t)          ----       67.48  29.23       80.99   3.90  11.09   
Max Entropy           ----       76.28  29.38       85.73   3.63  16.35   
Max Entropy(t=0.5)    ----       53.03  33.86       67.12   8.48   1.96   
Max Entropy(t=0.75)   ----       67.30  30.31       80.90   5.89   9.09   
Max Entropy(t=1.1)    ----       77.36  29.40       86.45   3.10  17.12   
Max Entropy (best t)  ----       77.36  29.38       86.45   3.10  17.12   
RbA                   ----       39.66  85.17       66.39   2.95  38.62   
RbA(t=0.5)            ----       49.28  81.57       89.26   2.30  46.35   
RbA(t=0.75)           ----       44.39  80.66       81.63   2.37  42.16   
RbA(t=1.1)            ----       36.78  86.73       59.22  97.59  36.90   
RbA (best t)          ----       49.28  80.66       89.26   2.30  46.35   
Max Logit             ----       57.12  90.30       77.03   2.37  37.87   
Max Logit(t=0.5)      ----       58.50  92.99       90.21   0.66  44.95   
Max Logit(t=0.75)     ----       57.39  91.43       85.37   0.74  41.34   
Max Logit(t=1.1)      ----       57.73  89.88       72.82   2.43  36.82   
Max Logit (best t)    ----       58.50  89.88       90.21   0.66  44.95   

                            FS Static        Road Anomaly         
                      FPR95     AuPRC  FPR95        AuPRC  FPR95  
Method                                                            
MSP                   59.63     55.21  27.59        66.55  37.25  
MSP(t=0.5)            43.02     19.33  42.40        41.23  46.75  
MSP(t=0.75)           46.23     38.07  33.46        57.27  40.88  
MSP(t=1.1)            63.33     59.42  25.67        68.51  35.87  
MSP (best t)          43.02     59.42  25.67        68.51  35.87  
Max Entropy           59.92     69.88  26.57        75.66  36.57  
Max Entropy(t=0.5)    42.91     30.11  41.86        54.31  47.23  
Max Entropy(t=0.75)   46.91     56.04  32.86        68.44  41.28  
Max Entropy(t=1.1)    63.10     72.09  24.49        77.34  34.54  
Max Entropy (best t)  42.91     72.09  24.49        77.34  34.54  
RbA                   47.86     84.44   3.04        55.57  72.77  
RbA(t=0.5)            35.54     90.88   2.92        62.64  33.23  
RbA(t=0.75)           54.27     89.27   2.74        60.03  67.44  
RbA(t=1.1)            50.74     81.70   3.92        52.99  74.92  
RbA (best t)          35.54     90.88   2.74        62.64  33.23  
Max Logit             20.65     86.52   7.46        61.26  35.30  
Max Logit(t=0.5)      25.52     89.05   7.51        62.80  42.61  
Max Logit(t=0.75)     24.38     88.46   7.57        61.94  37.30  
Max Logit(t=1.1)      19.84     85.46   7.42        60.85  37.03  
Max Logit (best t)    19.84     89.05   7.42        62.80  35.30

## 13. Qualitative best / worst anomaly figures (finetuned 40 ep vs EoMT Cityscapes)

Following the spirit of `Task4_Comparison.ipynb`, this section saves side-by-side
qualitative figures with **4 panels**:

`Input image | Anomaly GT | Finetuned 40 ep (heatmap) | EoMT Cityscapes (heatmap)`

For **each of the 5 anomaly datasets** and **each scoring method** (`MSP`,
`MaxLogit`, `Max Entropy`, `RbA`) the section selects the **best** and the
**worst** image and saves both. Ranking uses the **per-image AUPR (Average
Precision)** of the **finetuned 40 ep** model (FPR95 is unstable on a single
image, so it is reported but not used for selection). The `EoMT Cityscapes`
result is always drawn on the *same* scene for a fair comparison.

Each figure also carries a **data-grounded caption** explaining *why* the
finetuned model does well (on the best frame) or badly (on the worst frame),
derived from three cheap indicators computed on its score map:

- **localization precision `P@|A|`** — do its highest anomaly scores fall on the
  GT anomaly, or leak onto road/background (false positives)?
- **separation `z`** — how far the anomaly's mean score sits above the background
  (low ⇒ the anomaly is confidently absorbed into a known Cityscapes class);
- **anomaly size** — tiny anomalies make precision fragile.

Output: `results/task8/qualitative/` (40 PNGs) + a `selection.csv` summary with
metrics, the three indicators and the explanation text. This section is
**self-contained**: it builds its own models and does not touch the
checkpoints/evaluation defined above (it only reuses the helpers in
`evalAnomaly_eomt.py` and the `datasets` / path variables from the setup cells).


In [ ]:
# --- Qualitative best/worst: configuration and helpers ------------------------
# Self-contained: re-imports everything it needs and reuses evalAnomaly_eomt (E).

import os, glob, time
from types import SimpleNamespace
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from sklearn.metrics import average_precision_score
from ood_metrics import fpr_at_95_tpr

import evalAnomaly_eomt as E

QUAL_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
QUAL_USE_AMP = True
QUAL_TEMPERATURE = 1.0
QUAL_TARGET_SIZE = (E.IMG_HEIGHT, E.IMG_WIDTH)

# Anomaly-scoring methods to render (one best/worst pair per dataset x method).
QUAL_METHODS = ["msp", "maxlogit", "entropy", "rba"]
METHOD_LABEL = {"msp": "MSP", "maxlogit": "MaxLogit", "entropy": "Max Entropy", "rba": "RbA"}

# Best/worst are chosen by the per-image AUPR of this model. Both models are still
# drawn on the same selected scene.
QUAL_SELECTION_MODEL = "eomt_finetuned"

# The two models compared in every figure. `eomt_finetuned` = stage-2 weights
# trained for 40 epochs; `eomt_cityscapes` = the Cityscapes-trained checkpoint.
QUAL_CHECKPOINTS = {
    "eomt_finetuned": {
        "weights": WEIGHTS_ROOT / "finetuned" / "coco_to_cityscapes_stage2_unfreeze_last" / "stage2_weights_rerun.bin",
        "preset": "finetuned",
        "label": "Finetuned 40 ep",
    },
    "eomt_cityscapes": {
        "weights": WEIGHTS_ROOT / "eomt_cityscapes.bin",
        "preset": "cityscapes",
        "label": "EoMT Cityscapes",
    },
}

QUAL_DIR = RESULTS_ROOT / "qualitative"
QUAL_DIR.mkdir(parents=True, exist_ok=True)

print("Qualitative device:", QUAL_DEVICE)
print("Methods:", [METHOD_LABEL[m] for m in QUAL_METHODS])
print("Selection model:", QUAL_SELECTION_MODEL, "| metric: per-image AUPR")
print("Output dir:", QUAL_DIR)
for name, cfg in QUAL_CHECKPOINTS.items():
    print(f"  {name:16s} preset={cfg['preset']:10s} exists={cfg['weights'].exists()} -> {cfg['weights']}")


def qual_build_model(weights_path, preset):
    """Build + load an EoMT model the same way as evalAnomaly_eomt."""
    args = SimpleNamespace(
        preset=preset,
        num_blocks=3,
        patch_size=16,
        backbone_name="vit_base_patch14_reg4_dinov2",
    )
    args = E.apply_preset(args)
    model = E.build_eomt(args)
    model = E.load_eomt_weights(model, str(weights_path))
    return model.to(QUAL_DEVICE).eval()


def qual_method_score_map(method, ml, cl, sem_scores, sem_probs):
    """Return a per-pixel anomaly score map [H, W] as a numpy array (higher = more anomalous)."""
    if method == "rba":
        s = E.compute_rba_score(ml, cl, QUAL_TARGET_SIZE, temperature=QUAL_TEMPERATURE)
    else:
        s = E.compute_eomt_anomaly_score(sem_scores, sem_probs, method=method)
    return s.squeeze(0).float().cpu().numpy()


def qual_image_metrics(score_map, gt, valid):
    """Per-image AUPR and FPR95 on the valid (non-ignore) pixels."""
    label = gt[valid].astype(np.uint8)
    out = score_map[valid].astype(np.float32)
    aupr = float(average_precision_score(label, out))
    fpr95 = float(fpr_at_95_tpr(out, label))
    return aupr, fpr95


def qual_diagnostics(score_map, gt, valid):
    """Cheap, content-grounded indicators used to explain WHY a frame scores well/badly.

    - prec_at_k : localization precision. Among the |anomaly| highest-scored valid
                  pixels, the fraction that truly are anomalous. High => the model's
                  strongest anomaly signal lands on the object; low => it leaks onto
                  in-distribution regions (false positives).
    - z_gap     : (mean score on anomaly - mean score on normal) / std(normal).
                  How far the anomaly stands out above the background distribution.
                  Low => the anomaly is scored like a known class (absorbed into it).
    - anomaly_frac : anomaly pixels / valid pixels. Tiny anomalies make precision
                  fragile (a few stray high scores dominate).
    """
    s = np.asarray(score_map, dtype=np.float32)
    anom = s[gt == 1]
    norm = s[gt == 0]
    z_gap = float((anom.mean() - norm.mean()) / (norm.std() + 1e-6)) if anom.size and norm.size else 0.0
    anomaly_frac = float(anom.size) / float(max(int(valid.sum()), 1))

    vs = s[valid]
    vlabel = gt[valid]
    k = int((vlabel == 1).sum())
    if k > 0 and vs.size:
        topk = np.argsort(-vs)[:k]
        prec_at_k = float((vlabel[topk] == 1).sum()) / k
    else:
        prec_at_k = 0.0
    return {"prec_at_k": prec_at_k, "z_gap": z_gap, "anomaly_frac": anomaly_frac}


def qual_explanation(score_map, gt, valid, aupr, fpr95):
    """Build a short, data-grounded sentence explaining the finetuned model's result.

    Returns (text, diagnostics_dict).
    """
    d = qual_diagnostics(score_map, gt, valid)
    pk, z, frac = d["prec_at_k"], d["z_gap"], d["anomaly_frac"]

    if pk >= 0.6 and z >= 1.0:
        why = ("the finetuned model concentrates its highest anomaly scores on the anomalous "
               "object and clearly separates it from known classes -> high precision and AUPR.")
    elif pk < 0.4:
        why = ("the highest anomaly scores leak onto in-distribution regions (road/background), "
               "i.e. false positives, so precision collapses even if the object is partly detected.")
    elif z < 0.5:
        why = ("the anomaly is scored almost like the surrounding known classes (confidently "
               "absorbed into a Cityscapes category), so the score margin is too small to rank it on top.")
    else:
        why = ("detection is only partial: the object is somewhat highlighted but the score margin "
               "over the background is modest, mixing true and false positives.")

    if frac < 0.01:
        why += " The anomaly is very small, so a handful of stray high scores heavily penalise precision."

    text = (f"AUPR={aupr*100:.1f} | FPR95={fpr95*100:.1f} | localization P@|A|={pk:.2f} | "
            f"separation z={z:.1f} | anomaly size={frac*100:.1f}% -> {why}")
    return text, d


def _heatmap_for_display(score_map, valid):
    """Normalise a score map to [0, 1] using robust percentiles over valid pixels."""
    s = np.asarray(score_map, dtype=np.float32)
    v = s[valid]
    lo, hi = np.percentile(v, 1), np.percentile(v, 99)
    if not np.isfinite(hi) or hi <= lo:
        hi = lo + 1e-6
    return np.clip((s - lo) / (hi - lo), 0.0, 1.0)


def qual_save_figure(record, dataset_name, method, which, save_path, explanation):
    """4-panel figure + explanatory caption.

    Panels: Input | Anomaly GT | Finetuned heatmap | Cityscapes heatmap.
    """
    gt = record["gt"]
    valid = record["valid"]
    h, w = gt.shape

    gt_rgb = np.zeros((h, w, 3), dtype=np.float32)
    gt_rgb[gt == 255] = (0.35, 0.35, 0.35)   # ignore -> gray
    gt_rgb[gt == 1] = (1.0, 0.1, 0.1)        # anomaly -> red

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    axes[0].imshow(record["image"])
    axes[0].set_title("Input")

    axes[1].imshow(gt_rgb)
    axes[1].set_title("Anomaly GT (red)")

    ft = QUAL_CHECKPOINTS[QUAL_SELECTION_MODEL]
    im2 = axes[2].imshow(_heatmap_for_display(record["score_ft"], valid), cmap="inferno", vmin=0, vmax=1)
    axes[2].set_title(f"{ft['label']}\nAUPR={record['aupr_ft']*100:.1f}  FPR95={record['fpr_ft']*100:.1f}")
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    cs_name = next(n for n in QUAL_CHECKPOINTS if n != QUAL_SELECTION_MODEL)
    cs = QUAL_CHECKPOINTS[cs_name]
    im3 = axes[3].imshow(_heatmap_for_display(record["score_cs"], valid), cmap="inferno", vmin=0, vmax=1)
    axes[3].set_title(f"{cs['label']}\nAUPR={record['aupr_cs']*100:.1f}  FPR95={record['fpr_cs']*100:.1f}")
    fig.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

    for ax in axes:
        ax.axis("off")

    fig.suptitle(
        f"{dataset_name}  |  {METHOD_LABEL[method]}  |  {which.upper()} "
        f"(by {ft['label']} AUPR)  |  {os.path.basename(record['path'])}",
        fontsize=13,
    )
    # Data-grounded explanation of the finetuned model's behaviour on this frame.
    fig.text(0.5, -0.02, explanation, ha="center", va="top", fontsize=10, wrap=True,
             bbox=dict(boxstyle="round", facecolor="#f2f2f2", edgecolor="0.7"))
    fig.tight_layout()
    fig.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.close(fig)


print("Helpers ready.")


In [ ]:
# --- Qualitative best/worst: scan the datasets and save the figures ----------
# For every (dataset, method) we keep the image with the highest and the lowest
# per-image AUPR of the finetuned model, then render a 4-panel figure with both
# models on that same scene. One forward pass per model per image.

assert QUAL_CHECKPOINTS[QUAL_SELECTION_MODEL]["weights"].exists(), "Finetuned weights not found."
_cs_name = next(n for n in QUAL_CHECKPOINTS if n != QUAL_SELECTION_MODEL)
assert QUAL_CHECKPOINTS[_cs_name]["weights"].exists(), "Cityscapes weights not found."

print("Building models...")
model_ft = qual_build_model(QUAL_CHECKPOINTS[QUAL_SELECTION_MODEL]["weights"],
                            QUAL_CHECKPOINTS[QUAL_SELECTION_MODEL]["preset"])
model_cs = qual_build_model(QUAL_CHECKPOINTS[_cs_name]["weights"],
                            QUAL_CHECKPOINTS[_cs_name]["preset"])

summary_rows = []
saved_paths = []

for dataset_name, pattern in datasets.items():
    paths = sorted(glob.glob(str(pattern)))
    if not paths:
        print(f"[skip] {dataset_name}: no images")
        continue

    # selected[method] = {"best": record_or_None, "worst": record_or_None}
    selected = {m: {"best": None, "worst": None} for m in QUAL_METHODS}
    n_used = 0
    t0 = time.time()

    for path in paths:
        pathGT = E.get_gt_path(path)
        if not os.path.exists(pathGT):
            continue

        gt = E.convert_gt(np.array(E.target_transform(Image.open(pathGT))), pathGT)
        if 1 not in np.unique(gt):
            continue
        valid = (gt == 0) | (gt == 1)
        label = gt[valid]
        if label.min() == label.max():   # need both normal and anomaly pixels for AUPR
            continue

        img_t = E.input_transform(Image.open(path).convert("RGB")).unsqueeze(0).float().to(QUAL_DEVICE)

        with torch.no_grad():
            with torch.autocast(QUAL_DEVICE.type, dtype=torch.float16,
                                enabled=(QUAL_USE_AMP and QUAL_DEVICE.type == "cuda")):
                ml_ft_l, cl_ft_l = model_ft(img_t)
                ml_cs_l, cl_cs_l = model_cs(img_t)

            ml_ft, cl_ft = ml_ft_l[-1].float(), cl_ft_l[-1].float()
            ml_cs, cl_cs = ml_cs_l[-1].float(), cl_cs_l[-1].float()

            ss_ft, sp_ft = E.eomt_to_pixel_scores(ml_ft, cl_ft, QUAL_TARGET_SIZE, temperature=QUAL_TEMPERATURE)
            ss_cs, sp_cs = E.eomt_to_pixel_scores(ml_cs, cl_cs, QUAL_TARGET_SIZE, temperature=QUAL_TEMPERATURE)

        disp_img = img_t[0].permute(1, 2, 0).cpu().numpy()
        n_used += 1

        for method in QUAL_METHODS:
            s_ft = qual_method_score_map(method, ml_ft, cl_ft, ss_ft, sp_ft)
            s_cs = qual_method_score_map(method, ml_cs, cl_cs, ss_cs, sp_cs)

            aupr_ft, fpr_ft = qual_image_metrics(s_ft, gt, valid)
            aupr_cs, fpr_cs = qual_image_metrics(s_cs, gt, valid)

            record = {
                "path": path, "image": disp_img, "gt": gt, "valid": valid,
                "score_ft": s_ft, "score_cs": s_cs,
                "aupr_ft": aupr_ft, "fpr_ft": fpr_ft,
                "aupr_cs": aupr_cs, "fpr_cs": fpr_cs,
            }

            cur_best = selected[method]["best"]
            if cur_best is None or aupr_ft > cur_best["aupr_ft"]:
                selected[method]["best"] = record
            cur_worst = selected[method]["worst"]
            if cur_worst is None or aupr_ft < cur_worst["aupr_ft"]:
                selected[method]["worst"] = record

    # Render best/worst for this dataset, with a data-grounded explanation each.
    for method in QUAL_METHODS:
        for which in ("best", "worst"):
            rec = selected[method][which]
            if rec is None:
                continue

            explanation, diag = qual_explanation(
                rec["score_ft"], rec["gt"], rec["valid"], rec["aupr_ft"], rec["fpr_ft"]
            )

            out_path = QUAL_DIR / f"task8_qual_{dataset_name}_{method}_{which}.png"
            qual_save_figure(rec, dataset_name, method, which, out_path, explanation)
            saved_paths.append(out_path)

            summary_rows.append({
                "dataset": dataset_name,
                "method": METHOD_LABEL[method],
                "which": which,
                "image": os.path.basename(rec["path"]),
                "aupr_finetuned": round(rec["aupr_ft"] * 100, 2),
                "fpr95_finetuned": round(rec["fpr_ft"] * 100, 2),
                "aupr_cityscapes": round(rec["aupr_cs"] * 100, 2),
                "fpr95_cityscapes": round(rec["fpr_cs"] * 100, 2),
                "prec_at_k": round(diag["prec_at_k"], 3),
                "z_gap": round(diag["z_gap"], 2),
                "anomaly_frac_%": round(diag["anomaly_frac"] * 100, 2),
                "explanation": explanation,
                "file": out_path.name,
            })

    print(f"[done] {dataset_name}: {n_used} valid images, "
          f"{2 * len(QUAL_METHODS)} figures in {time.time() - t0:.1f}s")

# Free the models.
del model_ft, model_cs
if QUAL_DEVICE.type == "cuda":
    torch.cuda.empty_cache()

# Save and show the selection summary.
summary_df = pd.DataFrame(summary_rows)
summary_csv = QUAL_DIR / "selection.csv"
summary_df.to_csv(summary_csv, index=False)

print(f"\nSaved {len(saved_paths)} figures to: {QUAL_DIR}")
print("Selection summary:", summary_csv)
with pd.option_context("display.max_colwidth", 200):
    display(summary_df.drop(columns=["explanation"]))
    display(summary_df[["dataset", "method", "which", "image", "explanation"]])
